# 6. Evaluation

We compare the four methods on the same data and with the same scores. There is no correct answer to check against, so we look from several angles:

- **E1** internal scores: how tight and separated the clusters are
- **E2** same-product check: does a method merge features that one product lists separately?
- **E3** hand-labelled pairs: does a method agree with us?
- **E6** examples side by side
- **E7** general vs filtered spectral clustering

## 6.1 Setup

Each method is run once with its final setting, chosen in sections 4.1–5.1:

| Method | Setting |
|---|---|
| Agglomerative | average linkage, threshold 0.70 |
| DBSCAN | `eps` 0.40, `min_samples` 2 |
| HDBSCAN | `min_cluster_size` 2, `min_samples` 2 |
| Spectral | 15 neighbours, k = 35 |

DBSCAN and HDBSCAN leave some feature names as noise. To score all 445 feature names, each noise feature name counts as its own cluster, and **coverage** shows how many are actually clustered.

In [1]:
import csv
import json
import time
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import AgglomerativeClustering, DBSCAN, HDBSCAN, KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_score

with np.load("data/similarity.npz") as saved:
    S = saved["similarity"].astype(float)
    feature_names = saved["titles"].tolist()
products = json.loads(Path("data/products_en.json").read_text(encoding="utf8"))
feature_records = json.loads(Path("data/features_en.json").read_text(encoding="utf8"))["attribute_records"]

n = len(feature_names)
index_of = {name: i for i, name in enumerate(feature_names)}
D = np.clip(1 - S, 0, 1)
np.fill_diagonal(D, 0)
G = S.copy()
np.fill_diagonal(G, 0)

FINAL = {
    "agglomerative_threshold": 0.70,
    "dbscan_eps": 0.40,
    "dbscan_min_samples": 2,
    "hdbscan_min_cluster_size": 2,
    "hdbscan_min_samples": 2,
    "spectral_k": 35,
    "spectral_neighbors": 15,
    "seed": 0,
}
print(f"Loaded {n} feature names and {len(products)} products")

Loaded 445 feature names and 208 products


In [2]:
def full_labels(labels):
    """Give every noise feature name (-1) its own singleton cluster."""
    labels = np.asarray(labels).copy()
    noise = np.where(labels == -1)[0]
    start = labels.max() + 1 if len(noise) < len(labels) else 0
    labels[noise] = np.arange(start, start + len(noise))
    return labels


def mark_small_clusters_as_noise(labels, minimum_size=2):
    labels = np.asarray(labels).copy()
    for label in np.unique(labels):
        if label != -1 and np.count_nonzero(labels == label) < minimum_size:
            labels[labels == label] = -1
    return labels


def medoid_db_like(labels):
    cluster_ids = np.unique(labels)
    medoids, scatters = [], []
    for cluster_id in cluster_ids:
        members = np.flatnonzero(labels == cluster_id)
        medoid = members[np.argmin(D[np.ix_(members, members)].sum(axis=1))]
        medoids.append(medoid)
        scatters.append(D[members, medoid].mean())
    separation = D[np.ix_(medoids, medoids)].copy()
    np.fill_diagonal(separation, np.inf)
    scatters = np.array(scatters)
    ratios = (scatters[:, None] + scatters[None, :]) / np.maximum(separation, 1e-12)
    return float(ratios.max(axis=1).mean())


def modularity(labels):
    degree = G.sum(axis=1)
    two_m = degree.sum()
    same = labels[:, None] == labels[None, :]
    return float(((G - np.outer(degree, degree) / two_m) * same).sum() / two_m)


def build_affinity(similarity, k_neighbors):
    W = similarity.copy()
    np.fill_diagonal(W, 0)
    threshold = np.sort(W, axis=1)[:, -k_neighbors][:, None]
    return np.where((W >= threshold) | (W >= threshold.T), W, 0)


def spectral_eigen(similarity, k_neighbors):
    """Eigenvalues and eigenvectors of the normalized Laplacian, smallest first."""
    W = build_affinity(similarity, k_neighbors)
    inv_sqrt = 1 / np.sqrt(np.maximum(W.sum(axis=1), 1e-12))
    laplacian = np.eye(len(W)) - W * np.outer(inv_sqrt, inv_sqrt)
    return np.linalg.eigh(laplacian)


def spectral_from_eigenvectors(eigenvectors, k, seed=0):
    U = eigenvectors[:, :k]
    U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
    return KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(U)

In [3]:
results = {}

start = time.perf_counter()
labels = AgglomerativeClustering(n_clusters=None, metric="precomputed", linkage="average",
                                 distance_threshold=FINAL["agglomerative_threshold"]).fit_predict(D)
results["Agglomerative"] = {"labels": labels, "seconds": time.perf_counter() - start}

start = time.perf_counter()
labels = DBSCAN(eps=FINAL["dbscan_eps"], min_samples=FINAL["dbscan_min_samples"], metric="precomputed").fit_predict(D)
results["DBSCAN"] = {"labels": mark_small_clusters_as_noise(labels), "seconds": time.perf_counter() - start}

start = time.perf_counter()
labels = HDBSCAN(min_cluster_size=FINAL["hdbscan_min_cluster_size"], min_samples=FINAL["hdbscan_min_samples"],
                 metric="precomputed", copy=True).fit_predict(D)
results["HDBSCAN"] = {"labels": labels, "seconds": time.perf_counter() - start}

start = time.perf_counter()
_, eigenvectors = spectral_eigen(S, FINAL["spectral_neighbors"])
labels = spectral_from_eigenvectors(eigenvectors, FINAL["spectral_k"], FINAL["seed"])
results["Spectral"] = {"labels": labels, "seconds": time.perf_counter() - start}

for name, result in results.items():
    clusters = len(set(result["labels"].tolist()) - {-1})
    print(f"{name:14} {clusters:4} clusters, {np.mean(result['labels'] != -1):6.1%} coverage")

Agglomerative   138 clusters, 100.0% coverage
DBSCAN           59 clusters,  37.1% coverage
HDBSCAN         106 clusters,  73.3% coverage
Spectral         35 clusters, 100.0% coverage


## 6.2 Comparison of all methods

### E1. Internal scores

- **Silhouette** (higher is better): is a feature name closer to its own cluster than to the next one?
- **Davies–Bouldin, medoid version** (lower is better): are clusters compact compared with the distance between them?
- **Modularity** (higher is better): are links inside clusters stronger than chance?

These scores describe the shape of the clusters, not whether they make sense. For DBSCAN and HDBSCAN we also show the silhouette of the clustered feature names only.

In [4]:
rows = []
for name, result in results.items():
    labels = result["labels"]
    clustered = labels != -1
    all_labels = full_labels(labels)
    sizes = Counter(labels[clustered].tolist())
    silhouette_clustered = silhouette_score(D[np.ix_(clustered, clustered)], labels[clustered], metric="precomputed")
    rows.append({
        "method": name,
        "clusters": len(sizes),
        "largest": max(sizes.values()),
        "coverage": round(100 * clustered.mean(), 1),
        "silhouette": round(silhouette_score(D, all_labels, metric="precomputed"), 3),
        "silhouette_clustered_only": round(silhouette_clustered, 3),
        "medoid_db_like": round(medoid_db_like(all_labels), 3),
        "modularity": round(modularity(all_labels), 3),
        "runtime_s": round(result["seconds"], 3),
    })
internal_table = pd.DataFrame(rows).set_index("method")
internal_table

,clusters,largest,coverage,silhouette,silhouette_clustered_only,medoid_db_like,modularity,runtime_s
method,,,,,,,,
Agglomerative,138,20,100.0,0.184,0.184,0.911,0.070,0.028
DBSCAN,59,9,37.1,0.138,0.475,0.492,0.016,0.005
HDBSCAN,106,9,73.3,0.178,0.281,0.725,0.030,0.014
Spectral,35,24,100.0,0.137,0.137,1.585,0.104,0.081


### E2. Same-product check

A product never lists the same property twice. So two feature names on the same product are different properties and should not share a cluster.

For each method: of all pairs it puts in one cluster, how many appear together on a product? Lower is better. Read it together with coverage: a method that clusters little also makes few of these mistakes.

In [5]:
# Pairs of feature names (as matrix indices) that appear together on at least one product
same_product_pairs = set()
for product in products:
    indices = sorted(index_of[name] for name in set(product["titles"]))
    same_product_pairs.update(combinations(indices, 2))


def grouped_pairs(labels):
    """All pairs of feature names that a method puts in the same non-noise cluster."""
    pairs = set()
    for label in set(labels.tolist()) - {-1}:
        members = np.flatnonzero(labels == label)
        pairs.update(combinations(members.tolist(), 2))
    return pairs


rows = []
for name, result in results.items():
    pairs = grouped_pairs(result["labels"])
    conflicts = pairs & same_product_pairs
    rows.append({
        "method": name,
        "coverage": round(100 * np.mean(result["labels"] != -1), 1),
        "grouped_pairs": len(pairs),
        "same_product_pairs_grouped": len(conflicts),
        "conflict_rate_%": round(100 * len(conflicts) / len(pairs), 1),
    })
print(f"{len(same_product_pairs)} pairs of feature names appear together on a product")
same_product_table = pd.DataFrame(rows).set_index("method")
same_product_table

1442 pairs of feature names appear together on a product


,coverage,grouped_pairs,same_product_pairs_grouped,conflict_rate_%
method,,,,
Agglomerative,100.0,1302,11,0.8
DBSCAN,37.1,201,0,0.0
HDBSCAN,73.3,448,6,1.3
Spectral,100.0,2876,49,1.7


### E3. Hand-labelled pairs

`data/eval_pairs.csv` has 100 pairs of feature names, each labelled `same` (same property) or `not_same`. Pairs 0–23 come from `3_testing_similarity.ipynb`; the others are sampled pairs where the methods disagree, all agree, or never group two very similar names.

- **Precision:** of the labelled pairs a method groups, how many are `same`?
- **Recall:** of the `same` pairs, how many does the method group?

The file is only created if it does not exist, so labels are never overwritten.

In [6]:
PAIRS_PATH = Path("data/eval_pairs.csv")

REVIEWED_PAIRS = [  # from 3_testing_similarity.ipynb, to be re-checked
    ("High stretch and anatomically optimised fit", "Ultra-lightweight design", "not_same"),
    ("Premium high-stretch material back", "Invisible under a loose short", "not_same"),
    ("Silicone-enhanced palm sections", "Comfortable padded palms", "not_same"),
    ("Comfortable padded palms", "Insulating & Breathable", "not_same"),
    ("Aerodynamic dial coverage", "Bonded cuffs, hems, and high collar", "not_same"),
    ("Extra reinforcement", "Cordura®-reinforced shaft", "not_same"),
    ("High stretch and secure fit", "Silicone Hem Gripper", "same"),
    ("Windproof softshell 3-layer panels", "High-performance fabric with 10K water resistance and 40K breathability", "not_same"),
    ("Aerodynamic coverage", "YKK 2-way Vislon AquaGuard zipper", "not_same"),
    ("Road-specific fit", "Featherlight feel", "same"),
    ("All-season wear", "Premium merino wool blend", "not_same"),
    ("Silicone-sealed edges", "YKK 2-way waterproof zipper", "same"),
    ("Silicone palm grip", "Insulating & breathable", "not_same"),
    ("Odor-resistant lining", "Power-saving performance", "not_same"),
    ("Optimised finger split", "Pure merino wool", "not_same"),
    ("Extremely breathable", "Female-specific fit", "not_same"),
    ("Versatile style", "Soft sweat wiper", "not_same"),
    ("Compression shaft", "Raised micro-textured core panels", "not_same"),
    ("Insulating fleece inner", "Raw cut sleeves", "not_same"),
    ("Reinforced base", "No Itch Comfort", "not_same"),
    ("Thermal knit structure", "Zipped leg pocket", "not_same"),
    ("Zipperless comfort", "Enhanced grip", "not_same"),
    ("Windproof softshell fabric", "Subtle style", "not_same"),
    ("Comfortable mesh suspenders", "Tight fitting lid, no leaks", "not_same"),
]


def short_descriptions(name, limit=160):
    record = next(record for record in feature_records if record["title"] == name)
    text = " | ".join(dict.fromkeys(record["descriptions"])) or "(no description)"
    return text[:limit]


def build_pair_template(seed=42):
    rng = np.random.default_rng(seed)
    grouped = [grouped_pairs(result["labels"]) for result in results.values()]
    votes = Counter(pair for pairs in grouped for pair in pairs)
    reviewed = {tuple(sorted((index_of[a], index_of[b]))) for a, b, _ in REVIEWED_PAIRS}

    disagree = sorted(pair for pair, count in votes.items() if count < len(grouped) and pair not in reviewed)
    all_agree = sorted(pair for pair, count in votes.items() if count == len(grouped) and pair not in reviewed)
    upper_i, upper_j = np.triu_indices(n, k=1)
    order = np.argsort(-S[upper_i, upper_j])
    never_grouped = [(int(upper_i[o]), int(upper_j[o])) for o in order[:5000]
                     if (int(upper_i[o]), int(upper_j[o])) not in votes][:60]

    sampled = []
    for group, size, source in [(disagree, 46, "methods_disagree"), (all_agree, 15, "all_methods_group"),
                                (never_grouped, 15, "similar_never_grouped")]:
        for position in rng.choice(len(group), size=min(size, len(group)), replace=False):
            sampled.append((group[position], source))

    rows = [{"feature_name_a": a, "feature_name_b": b, "label": label, "source": "3_testing_similarity"}
            for a, b, label in REVIEWED_PAIRS]
    rows += [{"feature_name_a": feature_names[i], "feature_name_b": feature_names[j], "label": "", "source": source}
             for (i, j), source in sampled]
    with PAIRS_PATH.open("w", newline="", encoding="utf8") as file:
        writer = csv.DictWriter(file, fieldnames=["pair_id", "feature_name_a", "feature_name_b", "label", "labeller",
                                                  "source", "descriptions_a", "descriptions_b"])
        writer.writeheader()
        for pair_id, row in enumerate(rows):
            writer.writerow({"pair_id": pair_id, "labeller": "", **row,
                             "descriptions_a": short_descriptions(row["feature_name_a"]),
                             "descriptions_b": short_descriptions(row["feature_name_b"])})
    print(f"Created {PAIRS_PATH} with {len(rows)} pairs")


if not PAIRS_PATH.exists():
    build_pair_template()

pairs = pd.read_csv(PAIRS_PATH, keep_default_na=False)
print(pairs["source"].value_counts().to_string())
print(f"Labelled: {(pairs['label'] != '').sum()} of {len(pairs)}")

Created data/eval_pairs.csv with 100 pairs
source
methods_disagree         46
3_testing_similarity     24
all_methods_group        15
similar_never_grouped    15
Labelled: 24 of 100


In [7]:
labelled = pairs[pairs["label"].isin(["same", "not_same"])]
rows = []
for name, result in results.items():
    labels = result["labels"]
    tp = fp = fn = 0
    for a, b, label in zip(labelled["feature_name_a"], labelled["feature_name_b"], labelled["label"]):
        grouped = labels[index_of[a]] != -1 and labels[index_of[a]] == labels[index_of[b]]
        tp += grouped and label == "same"
        fp += grouped and label == "not_same"
        fn += (not grouped) and label == "same"
    precision = tp / (tp + fp) if tp + fp else float("nan")
    recall = tp / (tp + fn) if tp + fn else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else float("nan")
    rows.append({"method": name, "grouped_same": tp, "grouped_not_same": fp, "missed_same": fn,
                 "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)})
print(f"{len(labelled)} labelled pairs ({(labelled['label'] == 'same').sum()} same)")
pair_table = pd.DataFrame(rows).set_index("method")
pair_table

24 labelled pairs (3 same)


,grouped_same,grouped_not_same,missed_same,precision,recall,f1
method,,,,,,
Agglomerative,0,0,3,NaN,0.0,NaN
DBSCAN,0,0,3,NaN,0.0,NaN
HDBSCAN,0,0,3,NaN,0.0,NaN
Spectral,0,1,3,0.0,0.0,NaN


## 6.3 Qualitative examples

### E6. The same themes side by side

For four typical feature names, the cluster each method put them in.

In [8]:
THEMES = ["Windproof", "Waterproof protection", "Insulating", "Reflective details"]
SHOW = 8  # feature names listed per cluster

rows = []
for theme in THEMES:
    for name, result in results.items():
        labels = result["labels"]
        label = labels[index_of[theme]]
        if label == -1:
            members = [theme]
        else:
            members = [feature_names[i] for i in np.flatnonzero(labels == label)]
        others = [member for member in members if member != theme]
        shown = ", ".join(others[:SHOW]) + (f" (+{len(others) - SHOW} more)" if len(others) > SHOW else "")
        rows.append({"theme": theme, "method": name, "size": "noise" if label == -1 else len(members),
                     "grouped with": shown})

with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(rows).set_index(["theme", "method"]))

size  \
theme                 method                 
Windproof             Agglomerative     12   
                      DBSCAN             9   
                      HDBSCAN            9   
                      Spectral          12   
Waterproof protection Agglomerative     15   
                      DBSCAN             4   
                      HDBSCAN            4   
                      Spectral          16   
Insulating            Agglomerative     13   
                      DBSCAN         noise   
                      HDBSCAN            4   
                      Spectral          17   
Reflective details    Agglomerative      9   
                      DBSCAN             4   
                      HDBSCAN            5   
                      Spectral          10   

                                                                                                                                                                                                                                                                                           grouped with  
theme                 method                                                                                                                                                                                                                                                                             
Windproof             Agglomerative                              Windproof Front Panel, Windproof protection, Versatile protection, Windproof softshell 3-layer panels, Windtex Softshell 3-layer fabric, Windproof front panel, Windproof softshell fabric, Lightweight windproof protection (+3 more)  
                      DBSCAN                                             Windproof Front Panel, Windproof protection, Windproof softshell 3-layer panels, Windtex Softshell 3-layer fabric, Windproof front panel, Windproof softshell fabric, Windproof front panels, Deep winter windproof protection  
                      HDBSCAN                                            Windproof Front Panel, Windproof protection, Windproof softshell 3-layer panels, Windtex Softshell 3-layer fabric, Windproof front panel, Windproof softshell fabric, Windproof front panels, Deep winter windproof protection  
                      Spectral                           Windproof Front Panel, Windproof protection, Breathable, windproof fabric, Windproof softshell 3-layer panels, Windtex Softshell 3-layer fabric, Windproof front panel, Windproof softshell fabric, Lightweight windproof protection (+3 more)  
Waterproof protection Agglomerative  Waterproof and breathable, Full waterproof protection, Stretchy and waterproof, Waterproof and breathable insert, Waterproof (8K) and breathable (15K), Waterproof (8K) triple-layer fabric, Waterproof and windproof, Lightweight waterproof protection (+6 more)  
                      DBSCAN                                                                                                                                                                           Full waterproof protection, Lightweight waterproof protection, Weather protection 10K (10,000mm)  
                      HDBSCAN                                                                                                                                                                          Full waterproof protection, Lightweight waterproof protection, Weather protection 10K (10,000mm)  
                      Spectral                Waterproof and breathable, Full waterproof protection, Stretchy and waterproof, Waterproof (8K) and breathable (15K), Waterproof (8K) triple-layer fabric, Waterproof and windproof, Lightweight waterproof protection, High-performance fabric (+7 more)  
Insulating            Agglomerative                                                                     Light insulation, Thinsulate™ Insulation, Deep winter insulation, Primaloft Gold insulation, Deep winter protection, Cool wea

All four methods on the same 2D map (t-SNE, for display only). Grey crosses are noise.

In [9]:
embedding = TSNE(n_components=2, metric="precomputed", init="random", learning_rate="auto",
                 perplexity=30, random_state=0).fit_transform(D)

figure = make_subplots(rows=2, cols=2, subplot_titles=list(results))
for position, (name, result) in enumerate(results.items()):
    row, col = position // 2 + 1, position % 2 + 1
    labels = result["labels"]
    noise = labels == -1
    figure.add_trace(go.Scattergl(
        x=embedding[~noise, 0], y=embedding[~noise, 1], mode="markers",
        marker={"color": labels[~noise], "colorscale": "Turbo", "size": 7, "opacity": 0.8},
        text=np.array(feature_names)[~noise], hovertemplate="%{text}<extra></extra>", showlegend=False),
        row=row, col=col)
    if noise.any():
        figure.add_trace(go.Scattergl(
            x=embedding[noise, 0], y=embedding[noise, 1], mode="markers",
            marker={"color": "#999999", "size": 6, "symbol": "x", "opacity": 0.65},
            text=np.array(feature_names)[noise], hovertemplate="%{text}<extra>noise</extra>", showlegend=False),
            row=row, col=col)
figure.update_xaxes(showticklabels=False)
figure.update_yaxes(showticklabels=False)
figure.update_layout(template="simple_white", height=850, width=1100,
                     title="Feature-name clusters per method (shared t-SNE projection)")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

## 6.4 General vs filtered spectral clustering

### E7. Do the general clusters hold within a product type?

For each product type with at least 50 feature names, we cluster only its feature names (8 neighbours, k by eigengap) and compare with the general clusters (k = 35):

- **Purity:** does each filtered cluster come from one general cluster? Low means general themes are **merged**.
- **Inverse purity:** does each general cluster stay in one filtered cluster? Low means general themes are **split**.
- **ARI:** overall agreement, corrected for chance (1 = identical, 0 = chance).

Fewer clusters lower the purity by themselves, so we also compare at a **matched k**: the number of general clusters in that product type. The plot shows ARI for every k.

In [10]:
def purity(reference, predicted):
    """Share of items that belong to the most common reference cluster of their predicted cluster."""
    total = 0
    for label in np.unique(predicted):
        total += Counter(reference[predicted == label].tolist()).most_common(1)[0][1]
    return total / len(reference)


FILTER_NEIGHBORS, FILTER_K_MIN, MIN_FEATURE_NAMES = 8, 3, 50
general_labels = results["Spectral"]["labels"]

type_names = {}
for product in products:
    type_names.setdefault(product["product_type"], set()).update(product["titles"])

rows, sweeps = [], {}
for product_type, names in sorted(type_names.items(), key=lambda item: -len(item[1])):
    if len(names) < MIN_FEATURE_NAMES:
        continue
    indices = sorted(index_of[name] for name in names)
    S_type = S[np.ix_(indices, indices)]
    general = general_labels[indices]
    k_max = len(indices) // 5
    matched_k = len(set(general.tolist()))

    eigenvalues, eigenvectors = spectral_eigen(S_type, FILTER_NEIGHBORS)
    gaps = np.diff(eigenvalues)
    eigengap_k = int(np.argmax(gaps[FILTER_K_MIN - 1:k_max]) + FILTER_K_MIN)

    for label, k in [("eigengap", eigengap_k), ("matched", matched_k)]:
        filtered = spectral_from_eigenvectors(eigenvectors, k, FINAL["seed"])
        rows.append({"product type": product_type, "feature names": len(indices), "k from": label, "k": k,
                     "purity": round(purity(general, filtered), 3),
                     "inverse purity": round(purity(filtered, general), 3),
                     "ARI": round(adjusted_rand_score(general, filtered), 3)})
    sweeps[product_type] = {k: adjusted_rand_score(general, spectral_from_eigenvectors(eigenvectors, k, FINAL["seed"]))
                            for k in range(FILTER_K_MIN, max(k_max, matched_k) + 1)}

filtered_table = pd.DataFrame(rows).set_index(["product type", "k from"])
filtered_table

feature names   k  purity  inverse purity  \
product type          k from                                                
Cycling Wear Tops     eigengap             93   5   0.452           0.925   
                      matched              93  27   0.785           0.699   
Shoe Covers           eigengap             70   7   0.629           0.943   
                      matched              70  22   0.829           0.757   
Headwear              eigengap             65   4   0.369           0.954   
                      matched              65  22   0.831           0.800   
Cycling Wear Bottoms  eigengap             62   3   0.306           0.935   
                      matched              62  22   0.758           0.677   
Gloves Cycling Winter eigengap             58   6   0.431           0.948   
                      matched              58  22   0.845           0.776   
Socks                 eigengap             56   8   0.571           0.964   
                      matched              56  18   0.786           0.750   

                                  ARI  
product type          k from           
Cycling Wear Tops     eigengap  0.301  
                      matched   0.542  
Shoe Covers           eigengap  0.499  
                      matched   0.582  
Headwear              eigengap  0.216  
                      matched   0.585  
Cycling Wear Bottoms  eigengap  0.156  
                      matched   0.390  
Gloves Cycling Winter eigengap  0.316  
                      matched   0.547  
Socks                 eigengap  0.534  
                      matched   0.511

In [11]:
figure = go.Figure()
for product_type, scores in sweeps.items():
    figure.add_trace(go.Scatter(x=list(scores), y=list(scores.values()), mode="lines+markers", name=product_type))
figure.update_layout(template="simple_white", width=900, height=450, xaxis_title="k (filtered clusters)",
                     yaxis_title="ARI with general clusters", title="General vs filtered spectral clustering: agreement by k")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

## 6.5 Summary

_To be written once the E3 labels are in._